# Machine Learning Fundamentals

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 1/7

Instead of telling a computer every rule, we hand it examples and let it find the
pattern — this lesson maps the whole territory before we start coding seriously.

## 🎯 Learning Objectives

- Explain what machine learning is and how it differs from hand-written rules.
- Describe how AI, machine learning, and deep learning nest inside each other.
- Distinguish supervised, unsupervised, and reinforcement learning, with real applications of each.
- Use the core vocabulary fluently: features, labels, model, training, inference, parameters.
- Follow the canonical 7-step ML workflow from idea to deployed model.
- Load and explore scikit-learn's built-in datasets as pandas DataFrames.

## 1. What Is Machine Learning?

Traditional programming is **rules + data → answers**: a human writes every `if`.
Machine learning flips it to **data + answers → rules**: the model studies examples
and produces the rules itself.

Think of teaching a friend to spot spam. You would never recite grammar laws for
scammers — you would show them 50 real spam emails and 50 real ones until they
go *"oh, I see the pattern."* That is `.fit()`.

**Syntax:**
```python
from sklearn.some_module import SomeModel

model = SomeModel(hyperparameter=value)   # 1. choose the model family
model.fit(X_train, y_train)               # 2. learn patterns from examples
predictions = model.predict(X_new)        # 3. apply the learned patterns
```

In [ ]:
# Approach 1: HAND-WRITTEN RULES — a human decides what "looks like spam"
def rule_spam_filter(email_text):
    """Flag as SPAM when the email trips at least 2 suspicious keywords."""
    spam_words = ["free", "winner", "prize", "click here", "urgent"]
    text = email_text.lower()
    hits = sum(1 for word in spam_words if word in text)
    return "SPAM" if hits >= 2 else "HAM"

emails = [
    # true label shown as a comment
    "URGENT! You are a lucky WINNER, click here to claim your free prize",      # spam
    "Free coffee in the break room to celebrate the new office",                # ham
    "Team meeting moved to 3pm, please review the budget doc",                  # ham
    "Winner announcement: our design tool won first prize at the conference!",  # ham
]

for mail in emails:
    verdict = rule_spam_filter(mail)
    print(f"{verdict:<4} <- {mail[:65]}")

The last email is **not spam**, but it trips two keywords (`winner`, `prize`) — the
rule fires falsely. Hand-written rules are brittle: every fix creates new edge
cases, and spammers actively dodge known keywords. We need rules that **come from
the data**.

> 🔍 **Under the Hood:** a model is just a function with adjustable *numbers*
> called **parameters**. `.fit()` runs a search: try parameter values → measure
> how wrong the predictions are (a *loss*) → nudge the parameters to reduce the
> loss → repeat millions of times. After training, the "intelligence" is literally
> a small array of floats living in attributes like `model.coef_`. Prediction
> (`predict`) is plain arithmetic on those numbers — no magic, just tuned maths.

In [ ]:
# Approach 2: LEARNING the pattern from labelled examples (teaser — detail comes later)
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

train_emails = [
    "urgent click here claim your free prize now",
    "winner winner you were selected claim the reward",
    "free entry to win a brand new phone click here",
    "congratulations you are our lucky winner claim today",
    "cheap meds online no prescription buy now",
    "your account was locked please verify your identity now",   # sneaky spam, no classic words!
    "meeting notes are attached see you tomorrow",
    "lunch tomorrow at the usual place near the office",
    "please review the attached budget report",
    "the project deadline moved to friday next week",
    "can you send me the invoice copy when possible",
    "free coffee in the break room to celebrate the launch",
]
y = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]        # 1 = spam, 0 = ham

vectorizer = CountVectorizer()                   # turns text into word-count tables
X_train = vectorizer.fit_transform(train_emails)
clf = LogisticRegression().fit(X_train, y)       # <-- the learning happens HERE

new_emails = [
    "click here to claim your free prize winner",            # obvious spam
    "please verify your account identity now it was locked", # sneaky spam, dodges keywords
]
for mail in new_emails:
    rule_verdict = rule_spam_filter(mail)                    # reuse the rule-based filter
    pred = clf.predict(vectorizer.transform([mail]))[0]
    model_verdict = "SPAM" if pred == 1 else "HAM"
    print(f"rules -> {rule_verdict:<4} | model -> {model_verdict:<4} | {mail[:55]}")

See the second email? It contains **zero** classic spam keywords, so the hand-written
filter waves it through — but the model flags it, because during training it learned
that words like *verify*, *account*, *locked* travel together with spam. Nobody
programmed that. The data taught it.

## 2. AI ⊃ ML ⊃ Deep Learning

These three terms get used interchangeably in headlines. They are nested sets:

```
Artificial Intelligence   (any technique that makes machines act "smart")
└── Machine Learning      (learns the behaviour from data)
    ├── Classical ML      (this module — scikit-learn)
    └── Deep Learning     (module 14 — multi-layer neural networks)
```

| Field | What it is | Everyday examples |
|---|---|---|
| Artificial Intelligence | The broad goal: machines that behave intelligently | Route planning, chess engines, chatbots |
| Machine Learning | Subset: systems that improve from data without being explicitly programmed | Spam filters, price prediction, movie recommendations |
| Deep Learning | Subset of ML: many-layered neural networks that learn their own features | Face recognition, voice assistants, LLMs |

Every deep-learning system is machine learning; most machine learning is AI;
but plenty of AI (even old-school chess search) involves no learning at all.
This module stays in **classical ML with scikit-learn** — still the workhorse of
industry tabular problems, and the fastest way to internalise ideas that transfer
directly to deep learning.

## 3. Three Paradigms of Learning

Machine learning is usually sorted by *what kind of feedback the algorithm gets*:

| Paradigm | What you provide | What the model finds | Classic applications |
|---|---|---|---|
| Supervised — **regression** | Examples with a **number** answer (label) | A mapping from inputs to that number | House prices, temperature forecast, sales forecast |
| Supervised — **classification** | Examples with a **category** answer | Which category new inputs fall into | Spam vs ham, tumour benign/malignant, churn yes/no |
| Unsupervised — **clustering** | Examples with **no labels** | Natural groups in the data | Customer segments, topic groups |
| Unsupervised — **dimensionality reduction** | Unlabelled, many-column data | Fewer columns keeping the essence | Visualising 64-D digits, compression, noise removal |
| Reinforcement learning | An environment giving **rewards** | A policy: which action to take when | Game agents, robotics, RLHF for chat models |

Lessons 4–5 cover supervised learning; lesson 6 covers the unsupervised pair.

In [ ]:
# Same dataset, two paradigms: WITH labels vs WITHOUT labels
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)

# Supervised view: every flower arrives WITH its species label
print("Labelled classes:", dict(zip(iris.target_names.tolist(), [50, 50, 50])))
print(pd.Series(iris.target).value_counts().sort_index().to_dict())

# Unsupervised view: pretend the labels never existed
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
sizes = pd.Series(km.labels_).value_counts().sort_index().to_dict()
print("Groups discovered WITHOUT labels:", sizes)

No labels were given, yet KMeans recovered three groups whose sizes land close to
the true three species (50 / 50 / 50). That is the quiet thrill of unsupervised
learning — structure appears out of raw data.

## 4. Core Vocabulary

ML conversations use six words constantly. Learn them once, read any tutorial forever:

| Term | Meaning | In a housing-price project |
|---|---|---|
| **Feature** | An input column describing each example | Size in sqft, number of bedrooms |
| **Label (target)** | The answer we want to predict | Sale price in Taka |
| **Model** | The function that maps features → prediction | `ŷ = w·size + b` |
| **Training** | Adjusting the model using known answers | Fitting `w` and `b` on 500 sold houses |
| **Inference** | Using the trained model on new data | Estimating the price of a house not yet sold |
| **Parameters** | Numbers the model learned during training | The fitted values of `w` and `b` |

By convention: `X` = feature matrix (rows = examples, columns = features),
`y` = label vector. Almost every sklearn call starts from this pair.

In [ ]:
# Vocabulary made concrete with shapes
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)   # features
y = pd.Series(iris.target, name="species_code")           # labels

print("X shape (rows, columns):", X.shape)
print("y shape:", y.shape)
print("First example:", X.iloc[0].to_dict(), "->", y.iloc[0])
print("\n150 examples, each described by 4 features, each carrying 1 label.")

## 5. The 7-Step ML Workflow

Real projects follow a rhythm. Every remaining lesson slots into one of these steps:

| Step | Question it answers | Typical tools |
|---|---|---|
| 1. Frame the problem | What are we predicting, and what does "good" mean? | Pen and paper |
| 2. Collect & explore data | What does the data actually look like? | pandas, matplotlib |
| 3. Split the data | Can the model handle data it has NEVER seen? | `train_test_split`, CV |
| 4. Preprocess | Clean missing values, encode categories, scale features | Imputers, encoders, scalers |
| 5. Train | Which model captures the pattern? | `.fit()` |
| 6. Evaluate | How wrong is it — measured honestly? | metrics, `cross_val_score` |
| 7. Deploy & monitor | Does it survive contact with reality? | `joblib`, pipelines, MLOps |

Most beginner pain comes from skipping steps 1 and 3 — optimising the wrong thing,
on data the model already memorised.

In [ ]:
# The ENTIRE module in ~15 lines — a skeleton you will understand completely by lesson 7
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1-2. Problem: predict iris species from petal/sepal measurements
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# 3. Split BEFORE looking too closely at anything
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=42, stratify=y)
# 4. Preprocess (fitted on TRAIN only!)
scaler = StandardScaler().fit(X_train)
# 5. Train
knn = KNeighborsClassifier(n_neighbors=5).fit(scaler.transform(X_train), y_train)
# 6. Evaluate on held-out data
test_acc = accuracy_score(y_test, knn.predict(scaler.transform(X_test)))
print(f"Step 6 honest score: {test_acc:.2f}")
# 7. Deploy = save it (lesson 07 shows the joblib round-trip)

## 6. Exploring scikit-learn's Built-in Datasets

sklearn ships small classic datasets — perfect for practice because they download
nothing and load instantly. Each loader returns a **Bunch**: a dict-like object
with `.data`, `.target`, `.feature_names`, `.target_names`. Wrapping it in a
DataFrame makes it feel exactly like your own CSV.

**Syntax:**
```python
from sklearn.datasets import load_iris, load_diabetes

data = load_iris()
df = pd.DataFrame(data.data, columns=data.feature_names)  # features as a DataFrame
df["target"] = data.target                                # attach labels as a column
```

In [ ]:
# Explore IRIS — the "hello world" of classification (150 flowers, 3 species)
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species_code"] = iris.target

print("Shape:", df.shape)
print("Species names:", iris.target_names.tolist())
print(df.head())

In [ ]:
# Explore DIABETES — a regression dataset (442 patients, disease progression score)
import pandas as pd
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df["target"] = diabetes.target          # numeric severity -> regression!

print("Shape:", df.shape)
print("Features:", df.columns.tolist())
print("\nTarget summary:")
print(df["target"].describe().round(1))
print(df.head(3))

Notice the difference: iris `target` holds **codes for categories**
(classification), diabetes `target` holds **continuous numbers** (regression).
Same `.data`/`.target` structure either way.

## 7. Train/Test Intuition — a 60-Second Teaser

A model that merely **memorises** the training rows looks flawless on them and
stumbles on new rows — like a student who studied past exam papers word-for-word
and then meets a fresh question. So we always keep a **test set** the model never
sees during training, and score there. Full treatment: lesson 3.

In [ ]:
# Two models, one question: who GENERALISES?
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

models = {
    "KNN (k=3)":               KNeighborsClassifier(n_neighbors=3),
    "DecisionTree (unlimited)": DecisionTreeClassifier(random_state=42),
}
print(f"{'model':<26}{'train':>7}{'test':>7}")
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:<26}{model.score(X_train, y_train):>7.2f}"
          f"{model.score(X_test, y_test):>7.2f}")

Read the two columns like a doctor reads vitals. The unlimited tree scores a
perfect **1.00 at home** (it grew a branch for almost every training row) yet drops
on unseen flowers — that gap between the train column and the test column is the
single most important number in applied ML. We chase it for the rest of the module.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Evaluating on the training data | The score measures memorisation, not skill — often a flattering 99% | Split first; report test (or CV) scores |
| Tuning against the test set repeatedly | The test set quietly becomes training data; your "honest" number rots | Touch the test set once, at the very end |
| Jumping straight to fancy models | Data quality beats algorithm choice almost every time | Explore, clean, and establish a dumb baseline first |
| Collecting data with no thought to labelling | Garbage labels teach garbage patterns | Audit label quality before training |
| Treating ML as "import, fit, done" | Real cost lives in framing, data work, monitoring | Follow the 7-step workflow explicitly |

## 💡 Best Practices & Pro Tips

- **Baseline first:** predict the majority class (classification) or the mean
  (regression). Any real model must beat that bar to justify existing.
- **Fix your seeds** (`random_state=42`) so teammates reproduce your exact numbers.
- **Write the metric down before training.** "Accuracy" is not a goal;
  "recall on malignant cases ≥ 0.95" is.
- **Explore before modelling:** `.head()`, `.describe()`, one scatterplot — five
  minutes that routinely saves five days.
- **AI-engineering relevance:** even in the LLM era this discipline rules. RAG
  systems and fine-tunes are evaluated with held-out sets and baselines exactly
  like this, and classical sklearn models remain the cheap, interpretable choice
  behind countless production features.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `load_iris()` / `load_diabetes()` | Loads a built-in practice dataset (Bunch) | `data = load_iris()` |
| `pd.DataFrame(data.data, columns=data.feature_names)` | Bunch → tidy DataFrame | `df = pd.DataFrame(data.data, ...)` |
| `model.fit(X, y)` | Learns parameters from labelled examples | `clf.fit(X_train, y_train)` |
| `model.predict(X_new)` | Inference on unseen inputs | `clf.predict(X_test)` |
| `model.score(X, y)` | Quick default accuracy/R² | `clf.score(X_test, y_test)` |
| `train_test_split(X, y, test_size=0.3)` | Holds out data for honest scoring | See lesson 3 |

Key takeaways:
- ML learns the rules from **examples**; classic programming asks you to write them.
- Supervised = labelled answers (regression/classification); unsupervised = no labels
  (clustering/dimensionality reduction); reinforcement = learning from reward.
- Features go in `X`, labels in `y`; training tunes **parameters**; applying the
  result is **inference**.
- The train-vs-test gap exposes memorisation. Guard your test data like an exam paper.

## 🔗 Next Lesson

Continue to **[02_Data_Preprocessing](../02_Data_Preprocessing/notes.ipynb)** —
real data arrives messy: missing values, text categories, wildly different scales.
Let's clean it up properly.